# Predictive Modeling of Next-Day Rainfall for Kolkata Using Machine Learning
**Environmental Studies (EVS) — Topic 1: Climate Change & Weather Prediction**  
**Institution:** Techno India University, West Bengal  
**Data Source:** Copernicus Climate Change Service (ECMWF ERA5 Reanalysis) & IMD Standards  
**Target Location:** Kolkata ($22.57^\circ\text{N}, 88.36^\circ\text{E}$)  
**Algorithm:** Two-Stage Compound XGBoost (Classifier + Regressor)

## 1. Project Background & Environmental Significance
Kolkata experiences high vulnerability to sudden extreme downpours, especially during the Southwest Monsoon (June–September) and pre-monsoon convective thunderstorms (*Kalbaishakhi*). Unexpected downpours overwhelm municipal drainage sluices and cause debilitating urban waterlogging.

In this study, we build a **Two-Stage Machine Learning Pipeline**:
1. **Stage 1 (Classification):** Extreme Gradient Boosted Classifier predicting **Rain vs. No-Rain** for tomorrow ($t+1$).
2. **Stage 2 (Regression):** Extreme Gradient Boosted Regressor predicting the **precipitation intensity ($mm$)** on predicted rain days.

In [ ]:
# Step 1: Import Core Scientific & Machine Learning Libraries
import os
import csv
import math
import numpy as np
from datetime import datetime

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)
from xgboost import XGBClassifier, XGBRegressor
print('Environment initialized successfully!')

## 2. Load Processed Atmospheric Feature Dataset
The dataset contains 6,145 continuous daily records for Kolkata (2010–2026) with 34 engineered atmospheric physics features:
- **Thermodynamics:** Dew point depression, vapor pressure deficit, relative humidity, diurnal temperature range.
- **Barometric Pressure:** Surface pressure, 24h & 48h pressure tendencies (trough drops).
- **Kinematics:** $u$ and $v$ wind vectors (Bay of Bengal maritime moisture advection).
- **Autoregressive Memory:** Rain at $t, t-1, t-2, t-3, t-7$, cumulative 3-day & 7-day sums, rain streak counter.

In [ ]:
# Step 2: Load the Feature Matrix
csv_path = os.path.join('..', 'data', 'processed', 'rainfall_engineered_features.csv')
if not os.path.exists(csv_path):
    csv_path = os.path.join('data', 'processed', 'rainfall_engineered_features.csv')

with open(csv_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f'Total Daily Observations Loaded: {len(rows)}')
ignore = {'date', 'tomorrow_date', 'target_rain_tomorrow_binary', 'target_rain_tomorrow_amount'}
feature_cols = [c for c in reader.fieldnames if c not in ignore]
print(f'Number of Atmospheric Predictors: {len(feature_cols)}')

# Construct NumPy feature matrices
dates = np.array([r['date'] for r in rows])
X = np.array([[float(r[c]) for c in feature_cols] for r in rows], dtype=np.float32)
y_bin = np.array([int(r['target_rain_tomorrow_binary']) for r in rows], dtype=np.int32)
y_amt = np.array([float(r['target_rain_tomorrow_amount']) for r in rows], dtype=np.float32)
precip_today = np.array([float(r['precip_today']) for r in rows], dtype=np.float32)
print('Feature matrix shape:', X.shape)

## 3. Chronological Train-Test Split (Zero Data Leakage)
We strictly partition the timeline chronologically to prevent lookahead bias:
- **Training Window:** 2010 to 2023 (~5,106 days)
- **Holdout Test Window:** 2024 to November 2026 (~1,039 days of forward unseen reality)

In [ ]:
split_date = '2024-01-01'
train_mask = dates < split_date
test_mask = dates >= split_date

X_train, y_bin_train, y_amt_train = X[train_mask], y_bin[train_mask], y_amt[train_mask]
X_test, y_bin_test, y_amt_test = X[test_mask], y_bin[test_mask], y_amt[test_mask]

print(f'Training Samples: {len(X_train)} days')
print(f'Holdout Test Samples: {len(X_test)} days')

## 4. Benchmark Baseline Models
To prove the machine learning model is providing real scientific value, it must decisively beat the **Persistence Baseline** (*'Tomorrow will rain if it rained today'*).

In [ ]:
precip_today_test = precip_today[test_mask]
persist_preds = np.where(precip_today_test >= 2.5, 1, 0)

persist_acc = accuracy_score(y_bin_test, persist_preds)
persist_f1 = f1_score(y_bin_test, persist_preds, zero_division=0)
persist_mae = mean_absolute_error(y_amt_test, precip_today_test)

print(f'Persistence Baseline Accuracy: {persist_acc*100:.2f}%')
print(f'Persistence Baseline F1-Score: {persist_f1:.4f}')
print(f'Persistence Baseline MAE:      {persist_mae:.2f} mm')

## 5. Train Stage 1: XGBoost Rain Occurrence Classifier
We train an `XGBClassifier` using gradient-boosted decision trees with L1/L2 regularization and class-balance adjustment (`scale_pos_weight`).

In [ ]:
neg_count = np.sum(y_bin_train == 0)
pos_count = np.sum(y_bin_train == 1)
scale_pos_weight = neg_count / pos_count

clf = XGBClassifier(
    n_estimators=350,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=scale_pos_weight,
    reg_alpha=0.5,
    reg_lambda=1.5,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)
clf.fit(X_train, y_bin_train)

y_pred_proba = clf.predict_proba(X_test)[:, 1]
y_pred_bin = (y_pred_proba >= 0.5).astype(int)

clf_acc = accuracy_score(y_bin_test, y_pred_bin)
clf_auc = roc_auc_score(y_bin_test, y_pred_proba)
clf_f1 = f1_score(y_bin_test, y_pred_bin)

print('=' * 50)
print(f'XGBoost Classifier Accuracy: {clf_acc*100:.2f}% (vs Baseline: {persist_acc*100:.2f}%)')
print(f'XGBoost Classifier ROC-AUC:  {clf_auc:.4f}')
print(f'XGBoost Classifier F1-Score: {clf_f1:.4f}')
print('=' * 50)

## 6. Train Stage 2: XGBoost Precipitation Amount Regressor
On predicted rain days, we estimate the precipitation volume ($mm$) to identify potential urban waterlogging hazards ($>64.5\text{ mm}$ heavy rain threshold).

In [ ]:
rain_days_train = y_amt_train >= 0.1
reg = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=1.0,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=-1
)
reg.fit(X_train[rain_days_train], y_amt_train[rain_days_train])

# End-to-end compound inference
raw_reg_preds = reg.predict(X_test)
combined_amounts = np.where(y_pred_bin == 1, np.maximum(0.0, raw_reg_preds), 0.0)

compound_mae = mean_absolute_error(y_amt_test, combined_amounts)
error_reduction = ((persist_mae - compound_mae) / persist_mae) * 100

print('=' * 50)
print(f'Compound Model MAE:        {compound_mae:.2f} mm (vs Persistence: {persist_mae:.2f} mm)')
print(f'Error Reduction:           {error_reduction:.1f}% reduction in forecasting error')
print('=' * 50)

## 7. Atmospheric Feature Importance Analysis
We inspect the top physical drivers that influenced the tree splits to ensure scientific validity.

In [ ]:
importances = clf.feature_importances_
top_idx = np.argsort(importances)[::-1]

print('Top 10 Atmospheric Drivers in Decision Trees:')
for rank, idx in enumerate(top_idx[:10], start=1):
    print(f'{rank:2d}. {feature_cols[idx]:<25}: {importances[idx]*100:.2f}%')